## Restructure Attention Shift to have trials, sub-blocks and blocks

This script works with files of the form `_events_temp1.tsv` with columns
`onset`, `duration`, `sample`, `trial_type`, `value`, `event_code`, and `cond_code`.
The `key` is of the form `sub-xxx_run-y` which
uniquely specify each event file in the dataset.

**Transformations:**
1. Delete the `trial_type` and the `value` column.
2. Rename `repetition_type` as `rep_status` and `trigger` as `value`.
3. Insert a column called `trial` with the trial number. (Trial anchors are `show_face_initial`
and `show_cross`.  The excluded tags are `setup_left_sym` and `setup_right_sym`.
4. The `show_cross` value column should be 1.
5. Insert new column `rep_lag`.
6. Reorder the columns as `onset`, `duration`, `sample`, `event_type`, `face_type`,
`rep_status`, `rep_lag`, `value`, and `stim_file`.
7. Save as `*_events_temp2.tsv`

In [1]:
from hed.tools.hed_logger import HedLogger
from hed.tools.io_utils import get_file_list, make_file_dict
from hed.tools.data_utils import get_new_dataframe

def set_anchor_blocks(df, col_name, anchor_mask):
    total = 0
    anchor_list = list(anchor_mask.astype(int))
    for i, value in anchor_mask.iteritems():
        if value:
            total += 1
        anchor_list[i] = total
    df[col_name] = anchor_list
    return total

# Set up the logger
status = HedLogger()

# Make the dictionaries of the events.tsv files and the EEG.set events files
bids_root_path = 'G:\AttentionShift\AttentionShiftWorking'
bids_files = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix="_events_temp1")
bids_dict = make_file_dict(bids_files, indices=(0, 2))

trial_anchors = ['3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']
sub_anchors = ['1', '2']
for key, file in bids_dict.items():
    df = get_new_dataframe(file)

    # Set the number of trials
    trial_mask = df['event_code'].map(str).isin(trial_anchors)
    trial_count = set_anchor_blocks(df, 'trial', trial_mask)
    status.add(key, f"Added trial column: total trials {trial_count}")
    ## Set the sub blocks
    sub_mask = df['event_code'].map(str).isin(sub_anchors)
    sub_count = set_anchor_blocks(df, 'sub_block', sub_mask)
    status.add(key, f"Added sub_block column: total sub blocks {sub_count}")
    cond_code = 0
    total_cond = 0
    df['cond_block'] = 0
    for ind, row in df.iterrows():
        if row['cond_code'] != cond_code:
            total_cond += 1
            cond_code = row['cond_code']
        df.loc[ind, 'cond_block'] = total_cond
    status.add(key, f"{total_cond} condition blocks")

    df['sub_type'] = 'n/a'
    df['task_role'] = 'n/a'
    for sub_block in range(1,sub_count+1):
        df_sub = df.loc[df['sub_block'] == sub_blocks, :]

    for i in range(total_cond):
        sub_blocks = df.loc[df['cond_block'] == (i + 1), 'sub_block']
        sub_count = sub_blocks.max() - sub_blocks.min()
        sub_index = sub_blocks.index
        trials =  df.loc[df['cond_block'] == (i + 1), 'trial']
        trial_count = trials.max() - trials.min()
        cond_code = df.loc[df['cond_block'] == (i + 1), 'cond_code']
        status.add(key, f"Block {i + 1} [{sub_index.min()}, {sub_index.max()}] (cond_code {cond_code.iloc[0]}): {sub_count} subblocks  {trial_count} trials")

    filename = file[:-10] + "_temp2.tsv"
    df.to_csv(filename, sep='\t', index=False)
    status.add(key, f"Saved as _events_temp2.tsv")

In [2]:
status.print_log()

#df.loc[0, 'trial'] = 1


sub-001_run-01
	Added trial column: total trials 4793
	Added sub_block column: total sub blocks 479
	5 condition blocks
	Block 1 [0, 293] (cond_code 1): 23 subblocks  240 trials
	Block 2 [294, 582] (cond_code 2): 24 subblocks  240 trials
	Block 3 [583, 4130] (cond_code 3): 287 subblocks  2875 trials
	Block 4 [4131, 4991] (cond_code 1): 72 subblocks  718 trials
	Block 5 [4992, 5855] (cond_code 2): 72 subblocks  720 trials
	Saved as _events_temp2.tsv
sub-002_run-01
	Added trial column: total trials 4795
	Added sub_block column: total sub blocks 480
	5 condition blocks
	Block 1 [0, 288] (cond_code 1): 23 subblocks  240 trials
	Block 2 [289, 577] (cond_code 2): 23 subblocks  240 trials
	Block 3 [578, 4143] (cond_code 3): 287 subblocks  2878 trials
	Block 4 [4144, 5008] (cond_code 1): 71 subblocks  719 trials
	Block 5 [5009, 5873] (cond_code 2): 71 subblocks  718 trials
	Saved as _events_temp2.tsv
sub-003_run-01
	Added trial column: total trials 4795
	Added sub_block column: total sub block